# OpenClaw + Contextual AI — SEC Filing Sentinel

**Architecture:**
- **OpenClaw (Docker)** — Isolated container running daily web scraping via Brave Search, converts findings to PDFs, and pushes them to Contextual AI
- **Contextual AI** — Cloud datastore + RAG agent for storing, indexing, and querying scraped SEC documents
- **This notebook** — Sets up the backend (Part 1), builds the Docker image (Part 2), and queries the agent on-demand (Part 3)

---

## Part 1: Contextual AI Setup
Create the datastore and agent so OpenClaw has somewhere to push data and we have something to query.

In [1]:
import os
import re
from dotenv import load_dotenv
from contextual import ContextualAI

load_dotenv()

def update_env(key: str, value: str, env_path: str = ".env"):
    """Update a key in .env — replaces if exists, appends if not."""
    with open(env_path, "r") as f:
        content = f.read()
    pattern = rf"^{re.escape(key)}=.*$"
    if re.search(pattern, content, flags=re.MULTILINE):
        content = re.sub(pattern, f"{key}={value}", content, flags=re.MULTILINE)
    else:
        content = content.rstrip("\n") + f"\n{key}={value}\n"
    with open(env_path, "w") as f:
        f.write(content)

client = ContextualAI(api_key=os.environ["CONTEXTUAL_API_KEY"])
print("Contextual AI client initialized.")

Contextual AI client initialized.


### Create Datastore
This is where OpenClaw will push scraped SEC documents.

In [2]:
datastore = client.datastores.create(name="SEC Filing Monitor")
datastore_id = datastore.id
print(f"Datastore created: {datastore_id}")

update_env("DATASTORE_ID", datastore_id)
print("DATASTORE_ID saved to .env")

Datastore created: 1c19d80d-7281-4bba-beb8-503387357e2f
DATASTORE_ID saved to .env


### Create Agent
A Contextual AI agent backed by our datastore — this is what we query from Part 3.

In [3]:
agent = client.agents.create(
    name="SEC Filing Sentinel",
    description="Monitors and analyzes SEC filings, regulatory changes, and compliance risks.",
    datastore_ids=[datastore_id],
)
agent_id = agent.id
print(f"Agent created: {agent_id}")

update_env("AGENT_ID", agent_id)
print("AGENT_ID saved to .env")

Agent created: 0a7aed45-beef-419d-a6fe-2863e718db05
AGENT_ID saved to .env


---
## Part 2: OpenClaw in Docker
OpenClaw runs in an isolated container. It uses Brave Search to find SEC filings, converts them to PDFs, and uploads them to the Contextual datastore.

### Step 1: Build the Docker Image

In [22]:
!docker compose build --no-cache openclaw

[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.2s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 625B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 422B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.12-slim        0.2s
[+] Building 0.4s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 625B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 422B 

### Step 2: Launch OpenClaw
Start the container with API keys passed via .env. The container scrapes on a daily schedule and pushes documents to Contextual.

In [23]:
!docker compose up -d --build

# Verify it's running
!docker compose ps

[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.2s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 601B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 422B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.12-slim        0.2s
[+] Building 0.3s (3/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 601B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 422B 

### Step 3: Manual Scrape (Test Run)
Trigger a one-off scrape to verify the pipeline works end-to-end before relying on the daily schedule.

In [24]:
!docker compose exec openclaw python scrape.py --once

# Check what got uploaded
docs = client.datastores.documents.list(datastore_id=datastore_id)
print(f"\nDocuments in datastore: {len(docs.documents) if hasattr(docs, 'documents') else docs}")
for doc in (docs.documents if hasattr(docs, 'documents') else []):
    print(f"  - {doc.name} ({doc.id})")


--- Searching: site:sec.gov 10-K annual report 2026 ---
Traceback (most recent call last):
  File "/app/scrape.py", line 218, in <module>
    run_scrape()
  File "/app/scrape.py", line 163, in run_scrape
    results = brave_search(query)
              ^^^^^^^^^^^^^^^^^^^
  File "/app/scrape.py", line 99, in brave_search
    resp.raise_for_status()
  File "/usr/local/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.search.brave.com/res/v1/web/search?q=site%3Asec.gov+10-K+annual+report+2026&count=20&freshness=pw

Documents in datastore: 30
  - tmpcmq9xmze.pdf (b8f0bbb1-3d60-42ac-9267-9d2e66403f45)
  - tmpucnqd_i6.pdf (05c58f03-4757-4d17-8514-d7436a9ab01b)
  - tmp9e34g1zp.pdf (6bd9d03c-3cde-4f19-bcb5-1109fb5c03af)
  - tmp0p6_ap5r.pdf (01d34540-203d-48ca-a88c-bc9c0e631c60)
  - tmphb69yrox.pdf (a1815b3c-0b98-445e-b2a5-d05

### Step 4: Check Logs
View the container logs to confirm scraping and uploads are working.

In [25]:
!docker compose logs --tail 50 openclaw

openclaw-1  | /app/scrape.py:223: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
openclaw-1  |   print(f"\n[{datetime.utcnow().isoformat()}] Starting scheduled scrape...")


---
## Part 3: Query the Agent
Once OpenClaw has scraped and pushed documents, query the Contextual AI agent to analyze them.

In [28]:
# If resuming a session, reload IDs from .env
load_dotenv(override=True)
datastore_id = os.environ.get("DATASTORE_ID", datastore_id)
agent_id = os.environ.get("AGENT_ID", agent_id)

response = client.agents.query.create(
    agent_id=agent_id,
    messages=[{
        "role": "user",
        "content": "Summarize any recent SEC filings, regulatory changes, or compliance risks found in the ingested documents."
    }],
)

print("=== Agent Response ===")
print(response.message.content)

print("\n=== Sources ===")
if response.attributions:
    for attr in response.attributions:
        # Print whatever fields are available
        print(f"  - {attr}")
elif response.retrieval_contents:
    for rc in response.retrieval_contents:
        print(f"  - {rc}")

=== Agent Response ===
Recent SEC filings and regulatory disclosures include VEON's audited 2025 Form 20-F filing with the U.S. SEC, completed under Public Company Accounting Oversight Board standards for the year ended December 31, 2025.[7]()
This Form 20-F filing represents a significant compliance milestone, particularly with the PCAOB standards requirement.

| Company | Filing Type | Key Dates | Significant Details |
|---------|------------|-----------|--------------------|
| Corsair Gaming, Inc. | Form 8-K | March 11, 2026 (filed), March 5, 2026 (event) | Reported in a Form 8-K filed with the SEC, disclosing company information, stock listing, and compliance details.[1]() |
| Venture Global, Inc. | Form 8-K | March 13, 2026 | Includes official press release as Exhibit 99.1 and references risks detailed in annual report (Form 10-K for year ended December 31, 2024).[5,6]() |
| Intrepid Potash, Inc. | Form 8-K | March 11, 2026 | Appointed Cris Ingold, Chief Accounting Officer, as int

### Interactive Query
Ask OpenClaw anything about the scraped SEC filings.

In [30]:
load_dotenv(override=True)
agent_id = os.environ.get("AGENT_ID", agent_id)

# ✏️ Edit your question here:
question = "What companies filed 8-K forms this week and what were they about?"

response = client.agents.query.create(
    agent_id=agent_id,
    messages=[{"role": "user", "content": question}],
)

print(f"\n=== Question: {question} ===\n")
print(response.message.content)


=== Question: What companies filed 8-K forms this week and what were they about? ===

| Company | Filing Date | Description | 
|---------|-------------|-------------|
FedEx Corporation (NYSE: FDX) | Filed 1 week ago | Filed Form 8-K with the Securities and Exchange Commission providing update on key corporate governance and management events, as well as information about registered securities.[1]()
Colgate-Palmolive Company (NYSE: CL) | March 12, 2026 | Filed Form 8-K with the U.S. Securities and Exchange Commission.[3]()
Corsair Gaming, Inc. | March 11, 2026 | Filed Form 8-K reporting an event that occurred on March 5, 2026.[5]()
Innovex International, Inc. | March 11, 2026 | Filed Form 8-K with the Securities and Exchange Commission (SEC) to report on events as of March 5, 2026.[7]()
Maximus, Inc. | Filed 1 week ago | Filing specifically mentions continued engagement of KPMG LLP as auditor, supporting ongoing financial transparency and consistency in financial reporting.[9]()
Wheele

---
### Grounded RAG vs. General LLM — Side-by-Side Comparison
Our datastore now contains **180+ SEC filings** — 8-Ks, 10-Ks, 10-Qs, 20-Fs, and DEF 14As scraped directly from EDGAR. These queries show what happens when you ask analytical questions at scale.

ChatGPT (GPT-4o) is one of the most capable general LLMs available. But without access to the full corpus, it can't aggregate across 180+ documents, cite specific filings, or track your monitored companies. The SEC agent can — in seconds.

**The point:** This isn't about who can Google faster. It's about having a grounded, auditable system over YOUR data at scale.

In [ ]:
# Setup: initialize OpenAI client for comparison queries
from openai import OpenAI

load_dotenv(override=True)
agent_id = os.environ.get("AGENT_ID", agent_id)
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def ask_chatgpt(question: str) -> str:
    """Ask ChatGPT directly — no tools, no documents, just general knowledge."""
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        max_tokens=2048,
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content

def ask_sec_agent(question: str) -> str:
    """Ask the Contextual AI SEC agent — grounded in real filings."""
    response = client.agents.query.create(
        agent_id=agent_id,
        messages=[{"role": "user", "content": question}],
    )
    return response.message.content

print("Comparison functions ready. (ChatGPT vs. Contextual SEC Agent)")

#### Test 1: Specific company lookup across filings

In [ ]:
Q1 = "Did Goodyear, Super Micro Computer, or Faraday Future file anything with the SEC this week? What did they file and when?"

print("=" * 60)
print("GENERAL LLM (ChatGPT — no documents)")
print("=" * 60)
print(ask_chatgpt(Q1))

print("\n")
print("=" * 60)
print("SEC AGENT (Contextual AI — 180+ indexed filings)")
print("=" * 60)
print(ask_sec_agent(Q1))

#### Test 2: Executive changes across all recent filings

In [ ]:
Q2 = "Which companies replaced their CFO or appointed a new board member in the past week? I need the names, dates, and what happened."

print("=" * 60)
print("GENERAL LLM (ChatGPT — no documents)")
print("=" * 60)
print(ask_chatgpt(Q2))

print("\n")
print("=" * 60)
print("SEC AGENT (Contextual AI — 180+ indexed filings)")
print("=" * 60)
print(ask_sec_agent(Q2))

#### Test 3: Biotech/pharma filing activity

In [ ]:
Q3 = "How many biotech or pharmaceutical companies filed with the SEC this week? Which ones, and were any of them 10-K annual reports?"

print("=" * 60)
print("GENERAL LLM (ChatGPT — no documents)")
print("=" * 60)
print(ask_chatgpt(Q3))

print("\n")
print("=" * 60)
print("SEC AGENT (Contextual AI — 180+ indexed filings)")
print("=" * 60)
print(ask_sec_agent(Q3))

#### Test 4: Weekly compliance summary with citations

In [ ]:
Q4 = "I need to brief our compliance team on Monday. Summarize the most important SEC filings from this past week — focus on material events, leadership changes, and anything unusual. Cite the specific filings."

print("=" * 60)
print("GENERAL LLM (ChatGPT — no documents)")
print("=" * 60)
print(ask_chatgpt(Q4))

print("\n")
print("=" * 60)
print("SEC AGENT (Contextual AI — 180+ indexed filings)")
print("=" * 60)
print(ask_sec_agent(Q4))

---
## Part 4: Telegram Integration
Connect OpenClaw to Telegram so you can interact with it from your phone. We'll build this up step by step:

1. **4a: One-shot briefing** — Send a single SEC summary to Telegram
2. **4b: SEC chat bot** — Chat back and forth with the SEC agent via Telegram (runs in Docker, always on)
3. **4c: Full OpenClaw agent** — Upgrade the bot with ChatGPT + web search + SEC filings (the full agent experience)

### 4a: One-Shot Briefing
The simplest integration — query the SEC agent and push the summary to your phone. Good for daily briefings.

In [35]:
import requests as req

load_dotenv(override=True)
TELEGRAM_BOT_TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.environ.get("TELEGRAM_CHAT_ID")

def send_telegram(message: str):
    """Send a message via Telegram bot."""
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    resp = req.post(url, json={"chat_id": TELEGRAM_CHAT_ID, "text": message, "parse_mode": "Markdown"})
    resp.raise_for_status()
    print("Telegram message sent!")

# Query the agent and send the summary to Telegram
response = client.agents.query.create(
    agent_id=agent_id,
    messages=[{"role": "user", "content": "Give a brief summary of the most important SEC filings from today's scrape. Keep it under 500 words."}],
)

send_telegram(f"🔍 *OpenClaw SEC Briefing*\n\n{response.message.content}")

Telegram message sent!


### 4b: SEC Chat Bot
Now let's make it interactive — chat back and forth with the SEC agent via Telegram. This runs inside Docker so it's always on, even when the notebook is closed. It only talks to your Contextual SEC agent (no web search, no ChatGPT).

Run this cell to launch the SEC chat bot, then message your Telegram bot with any SEC question.

In [36]:
# Launch the SEC-only chat bot in Docker
# (stops the full agent if it was running — only one bot can poll Telegram at a time)
!docker compose up -d --build openclaw
!docker compose run -d --name sec-chat-bot openclaw python telegram_sec_bot.py

print("\nSEC chat bot is live! Message your Telegram bot.")
print("To stop: docker compose stop sec-chat-bot")

[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.2s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 601B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 472B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.12-slim        0.2s
[+] Building 0.4s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 601B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 472B 

### 4c: Full OpenClaw Agent
The final evolution — ChatGPT acts as the brain and decides which tools to use. The bot can now:
- Query your SEC filing database
- Search the live web via Brave
- Answer general questions from ChatGPT's own knowledge
- Combine multiple tools in a single answer

Once running, close the notebook — the bot stays alive as long as Docker is running.

**Try these in Telegram:**

| Question | What happens |
|----------|-------------|
| "What 8-K filings came in this week?" | Queries your SEC datastore |
| "What's Tesla's stock price right now?" | Searches the live web via Brave |
| "Compare NVDA and AMD market caps" | Web search for current data |
| "Any new proxy statements in our database?" | Queries SEC datastore |
| "Summarize the latest news about the SEC and crypto regulation" | Web search for current news |
| "What is a 10-K filing?" | Answers directly from ChatGPT's knowledge |
| "Find recent executive changes in our 8-K filings and look up those companies' stock performance" | Uses both tools — SEC filings + web search |

In [37]:
# Stop the SEC-only bot (only one bot can poll Telegram at a time)
!docker stop sec-chat-bot 2>/dev/null; docker rm sec-chat-bot 2>/dev/null

# Rebuild and launch both services (scraper + full agent)
!docker compose build --no-cache
!docker compose up -d
!docker compose ps

# Check the bot started
print("\n--- Full agent bot logs ---")
!docker compose logs --tail 10 telegram-bot

sec-chat-bot
sec-chat-bot
[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.2s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 1.18kB                                           0.0s
 => [openclaw internal] load build definition from Dockerfile              0.0s
 => => transferring dockerfile: 472B                                       0.0s
 => [openclaw internal] load metadata for docker.io/library/python:3.12-s  0.2s
[+] Building 0.4s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 1.18kB                                           0.0s
 => [openclaw internal] load build definition from Dockerfile              0.0s
 => => tran